# INRs

In [ ]:
import os

import matplotlib
import matplotlib.pyplot as plt
import skimage
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Compose, Normalize, Resize, ToTensor
import torch.nn as nn
import mon

mon.preload()
torch.manual_seed(0)

Constants:

In [ ]:
current_dir = mon.Path(os.getcwd())
root_dir    = current_dir.parents[0]
data_dir    = root_dir / "data"
run_dir     = root_dir / "run"

image_name  = ""
image_dir   = data_dir  / "sample"
image_file  = image_dir / f"{image_name}.jpg"
output_dir  = run_dir   / "inr"
output_dir.mkdir(parents=True, exist_ok=True)

iters  = 8000
imgsz  = 256
device = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device)

# For visualization
font_size     = 10
line_width    = 2.0
fig_size      = (5, 4.5)
matplotlib.rc("font", **{
	# "family" : "normal",
	"size"   : font_size
})
plt.rcParams["figure.figsize"]    = [5, 4.5]
plt.rcParams["figure.autolayout"] = True

Load images:

In [ ]:
# image  = get_cameraman(imgsz).to(device)
# coords = mon.nn.create_coords(imgsz).permute(2, 0, 1).unsqueeze(0).to(device)
# print(image.shape, coords.shape)

Define dataloader:

In [ ]:
def get_cameraman(imgsz: int = 256) -> torch.Tensor:
    image     = Image.fromarray(skimage.data.camera())
    transform = Compose([
        Resize(imgsz),
        ToTensor(),
        Normalize(torch.Tensor([0.5]), torch.Tensor([0.5]))
    ])
    image = transform(image)
    return image

In [ ]:
class CameramanDataset(Dataset):

    def __init__(self, imgsz: int, device: torch.device, channels_first: bool = True):
        super().__init__()
        if channels_first:
            self.image  = get_cameraman(imgsz).to(device)
            self.coords = mon.nn.create_coords(imgsz).permute(2, 0, 1).to(device)
        else:
            self.image  = get_cameraman(imgsz).permute(1, 2, 0).to(device)
            self.coords = mon.nn.create_coords(imgsz).to(device)

    def __len__(self):
        return 1

    def __getitem__(self, idx: int):
        if idx > 0:
            raise IndexError
        return self.image, self.coords

Training loop:

In [ ]:
def train(model, dataloader, iters, log_interval=10):
    optim = torch.optim.Adam(lr=2e-4, params=model.parameters())

    image, coords = next(iter(dataloader))
    print(image.shape, coords.shape)

    losses = []
    preds  = []
    for i in range(iters):
        output = model(coords)
        # loss   = nn.MSELoss()(image, output)
        loss   = nn.MSELoss()(image.view(1, -1, 1), output.view(1, -1, 1))

        if i % log_interval == 0:
            print(f"Iter {i:05d} | Loss {loss.item():.4f}")
            losses.append(loss.item())
            preds.append(output.detach().cpu())

        optim.zero_grad()
        loss.backward()
        optim.step()
    torch.cuda.empty_cache()

    return losses, preds

Define networks:

In [ ]:
siren    =   mon.nn.SIREN(2, 1, 256, hidden_layers=2).to(device)
conv_inr = mon.nn.ConvINR(2, 1, 32 , hidden_layers=10, kernel_size=3).to(device)

Fitting an image:

In [ ]:
dataset1    = CameramanDataset(imgsz, device, channels_first=False)
dataset2    = CameramanDataset(imgsz, device, channels_first=True)
dataloader1 = DataLoader(dataset1, batch_size=1, num_workers=0)
dataloader2 = DataLoader(dataset2, batch_size=1, num_workers=0)

In [ ]:
losses = {}
preds  = {}

In [ ]:
losses["siren"], preds["siren"] = train(siren, dataloader1, iters)

In [ ]:
losses["conv_inr"], preds["conv_inr"] = train(conv_inr, dataloader2, iters)

Visualize:

In [ ]:
def draw_loss(losses: dict[str, list]):
    plt.subplots(figsize=fig_size)
    for k, v in losses.items():
        plt.plot(v, label=k, linewidth=line_width)
        plt.legend(prop={"size": font_size})
    plt.xlabel("Steps")
    plt.ylabel("Loss")
    # plt.ylim(0.0, 1.0)
    plt.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    plt.legend(bbox_to_anchor=(1, 1), loc=1, borderaxespad=0)
    plt.savefig(f"{output_dir}/loss_{iters}.jpg", dpi=100, bbox_inches="tight")

In [ ]:
def draw_figure(image, text, save_path, cmap="gray"):
	font = {
		# 'family': 'serif',
		"color" : "white",
		"weight": "bold",
		"size"  : 28,
	}
	fig, axs       = plt.subplots(1, 1)
	dpi            = 100
	left, width    = 0, 1
	bottom, height = 0, 1
	right          = left   + width
	center_x       = left   + width  / 2
	center_y       = bottom + height / 2
	top            = bottom + height
	p              = plt.Rectangle((left, bottom), width, height, linewidth=0, fill=False, facecolor="none", edgecolor=None)
	p.set_transform(axs.transAxes)
	p.set_clip_on(True)

	axs.add_patch(p)
	axs.imshow(image, cmap=cmap)
	axs.title.set_text("")
	axs.text(right - 0.01, bottom + 0.01, text,
			 horizontalalignment = "right",
			 verticalalignment   = "bottom",
			 transform           = axs.transAxes,
			 fontdict            = font)
	axs.set_xticks([])
	axs.set_yticks([])
	axs.set_axis_off()
	plt.show()
	fig.savefig(save_path, dpi=dpi, bbox_inches="tight", pad_inches=0)

In [ ]:
def save_pred(preds: dict[str, list], iters: int, output_dir: mon.Path):
    for k, v in preds.items():
        draw_figure(v[-1].squeeze().numpy(), k, f"{output_dir}/{k}_{iters}.jpg")

In [ ]:
# draw_loss(losses)
save_pred(preds, iters, output_dir)